# LLaMA

---

## 一、Rotary Positional Encoding

假设$d$维嵌入向量为$x$, token位置为$m$, 旋转矩阵为$\mathbf{R}_{\theta, m}^d$, 则经过旋转位置编码的向量$\mathbf{R}_{\theta, m}^d x$为：
$$
\mathbf{R}_{\theta, m}^d x = 
\begin{bmatrix}
x_1 \\ x_2 \\ x_3 \\ x_4 \\ \dots \\ x_{d-1} \\ x_d
\end{bmatrix} \otimes 
\begin{bmatrix}
\cos(m\theta_1) \\ \cos(m\theta_1) \\ \cos(m\theta_2) \\ \cos(m\theta_2) \\ \dots \\ \cos(m\theta_{d/2}) \\ \cos(m\theta_{d/2})
\end{bmatrix} +
\begin{bmatrix}
-x_2 \\ x_1 \\ -x_4 \\ x_3 \\ \dots \\ -x_{d} \\ x_{d-1}
\end{bmatrix} \otimes
\begin{bmatrix}
\sin(m\theta_1) \\ \sin(m\theta_1) \\ \sin(m\theta_2) \\ \sin(m\theta_2) \\ \dots \\ \sin(m\theta_{d/2}) \\ \sin(m\theta_{d/2})
\end{bmatrix}
$$
其中$\theta_i = 10000^{-2(i-1)/d}$, $d$维数被划分为相邻两两一组。这种计算方法**比直接进行旋转矩阵乘法快得多**。

In [ ]:
import torch

"""
旋转位置编码: 一种相对位置编码，通过对原始向量进行旋转操作得到位置编码
对于一个query向量Wq*x, 通过对其乘以一个**旋转矩阵R**得到对应的旋转位置编码 
"""
def precompute_theta_pos_frequencies(dim: int, seq_len: int, device: str):
    assert dim % 2 == 0, "dimension must be even"
    
    """
    theta.shape: (dim/2)
    """
    i_iter = torch.arange(0, dim, 2).float()
    theta = 1.0 / (10000 ** (i_iter / dim)).to(device)
    
    """
    m.shape(position): (seq_len)
    freqs: (seq_len) @ (dim/2) -> (seq_len, dim/2), 即每个位置m都和所有的theta组合相乘, 得到正余弦内的数值
    """
    m = torch.arange(seq_len).to(device)
    freqs = torch.outer(m, theta).float()
    
    """
    torch.polar: 构建一个复数张量, 其元素模长均为1, 其元素角度来自freqs, 即复数为cos(freq)+i*sin(freq)
    freqs_complex.shape: (seq_len, dim/2)
    """
    freqs_complex = torch.polar(torch.ones_like(freqs), freqs)
    return freqs_complex

那么如何将一个输入嵌入向量$x$按公式(1)转换为带有位置编码信息的旋转向量呢？假设$d = 4$。

1. 借助复数向量, 将输入$x$的维度**相邻两两组合**, 维度变成dim/2 (与freq的维度保持一致)：
$$
\begin{bmatrix}
x_1 \\ x_2 \\ x_3 \\ x_4
\end{bmatrix} \Rightarrow \begin{bmatrix}
x_1 + i x_2 \\ x_3 + i x_4
\end{bmatrix}
$$
- 另外由上面的freq复数化得到的复数张量为:
$$
\begin{bmatrix}
\cos(m\theta_1) + i \sin(m\theta_1) \\
\cos(m\theta_2) + i \sin(m\theta_2) \\
\end{bmatrix}
$$

2. 将上述两个张量进行**逐元素相乘**, 得到旋转后的复数矩阵:
$$
\begin{bmatrix}
x_1 \cos(m\theta_1) - x_2 \sin(m\theta_1) + i [x_2 \sin(m\theta_1) + x_1\cos(m\theta_1)] \\
x_3 \cos(m\theta_2) - x_4 \sin(m\theta_2) + i [x_4 \sin(m\theta_2) + x_3\cos(m\theta_2)] \\
\end{bmatrix}
$$

3. 复向量**实数化**, 并拉直:
$$
\begin{bmatrix}
x_1 \cos(m\theta_1) - x_2 \sin(m\theta_1) \\
x_2 \cos(m\theta_1) + x_1 \sin(m\theta_1) \\
x_3 \cos(m\theta_2) - x_4 \sin(m\theta_2) \\
x_4 \cos(m\theta_2) + x_3 \sin(m\theta_2) \\
\end{bmatrix}
$$
- 可见上述公式与开头旋转编码定义的公式计算结果完全相同, 也是代码实际实现的流程。

In [ ]:
def apply_rotatary_embeddings(x: torch.Tensor, freqs_complex: torch.Tensor, device: str):
    """
    先将x的最后一维扩成2个 (对应一组实数+复数)
        (batch, seq_len, n_heads, dim) -> (batch, seq_len, n_heads, dim/2, 2)
    再将x转化为复数, 与freqs_complex形状相同
        (batch, seq_len, n_heads, dim/2, 2) -> (batch, seq_len, n_heads, dim/2)
    """
    x_complex = torch.view_as_complex(x.float().reshape(*x.shape[:-1], -1, 2))
    
    """
    先扩展出批次和多头的维度, 便于与x_complex逐元素相乘
        (seq_len, dim/2) -> (1, seq_len, 1, dim/2)
    再与x_complex逐元素相乘
        (1, seq_len, 1, dim/2) * (batch, seq_len, n_heads, dim/2) -> (batch, seq_len, n_heads, dim/2)
    """
    freqs_complex = freqs_complex.unsqueeze(0).unsqueeze(2)
    x_rotated = x_complex * freqs_complex
    
    """
    先添加末尾2维度, 还原回实数部分
        (batch, seq_len, n_heads, dim/2, 2)
    再拉直, 还原成输入x的形状 
        (batch, seq_len, n_heads, dim)
    """
    x_out = torch.view_as_real(x_rotated)
    x_out = x_out.reshape(*x.shape)
    
    return x_out.type_as(x).to(device)